In [1]:
from mipt import *
import numpy as np
import importlib
import gilbert

importlib.reload(gilbert)

from gilbert import HybridUQNBank3Q, gilbert_uqn_distance_3q

bank = HybridUQNBank3Q.load(
    "uqn_hybrid_bank.pt",
    device="cuda",
    dtype="complex64",
)

backend = AerSimulator(device="CPU", method="matrix_product_state")

n = 8
d = 2 * n
p_values = np.linspace(0.0, 1.0, 21)
circuit_reps = 10

means = []
stderrs = []

for p in p_values:
    Ds = []
    print(f"Simulating p={p:.3f}...")

    for rep in range(circuit_reps):
        # New unitary disorder and new measurement-location disorder every rep.
        qc = random_mipt_1d(n=n, d=d, p=float(p), closed=True)

        qc.save_density_matrix(
            qubits=[0, 1, 2],
            label="final_state",
            pershot=True,
        )

        tqc = transpile(qc, backend)

        # One conditional outcome trajectory for this independently drawn circuit.
        result = backend.run(tqc, shots=1).result()
        sub_m = result.data(0)["final_state"][0]

        res = gilbert_uqn_distance_3q(
            sub_m,
            bank=bank,
            input_order="qiskit",
            qiskit_qubit_order=(0, 1, 2),

            max_iter=36,
            memory=24,
            topk_per_family=2,
            fresh_starts_per_family=0,
            fast_families_to_refine=1,
            fast_refinement_steps=8,
            families_to_refine=4,
            refinement_steps=24,
            full_refine_every=8,
            memory_update_every=4,
            local_cache_size=24,

            bs_max_iter=80,
            bs_memory=40,
            bs_restarts=4,
            bs_sweeps=8,
            bs_initial_random_atoms=8,

            cache_refined_atoms=False,
            seed=100000 * rep + int(round(10000 * p)),
        )

        Ds.append(res.distance)

    Ds = np.asarray(Ds, dtype=float)
    mean = float(np.mean(Ds))
    stderr = float(np.std(Ds, ddof=1) / np.sqrt(len(Ds)))

    means.append(mean)
    stderrs.append(stderr)

    print(f"p={p:.3f}, D={mean:.8g} ± {stderr:.3g}")

Simulating p=0.000...
p=0.000, D=3.5323057e-05 ± 1.48e-05
Simulating p=0.050...
p=0.050, D=0.0032626148 ± 0.000853
Simulating p=0.100...
p=0.100, D=0.01012164 ± 0.00283
Simulating p=0.150...
p=0.150, D=0.0069225606 ± 0.00176
Simulating p=0.200...
p=0.200, D=0.015162677 ± 0.00849
Simulating p=0.250...
p=0.250, D=0.01366766 ± 0.00512
Simulating p=0.300...
p=0.300, D=0.032099374 ± 0.0223
Simulating p=0.350...
p=0.350, D=0.0091212916 ± 0.0034
Simulating p=0.400...
p=0.400, D=0.0044692397 ± 0.00062
Simulating p=0.450...
p=0.450, D=0.0042547483 ± 0.000793
Simulating p=0.500...
p=0.500, D=0.0040800962 ± 0.000961
Simulating p=0.550...
p=0.550, D=0.0034050124 ± 0.000974
Simulating p=0.600...
p=0.600, D=0.0027299988 ± 0.00048
Simulating p=0.650...
p=0.650, D=0.0020464185 ± 0.000645
Simulating p=0.700...
p=0.700, D=0.0027576173 ± 0.000726
Simulating p=0.750...
p=0.750, D=0.002640162 ± 0.000979
Simulating p=0.800...
p=0.800, D=0.001351871 ± 0.000396
Simulating p=0.850...
p=0.850, D=0.00079939345 ±

In [1]:
from mipt import *
from gnme import random_pure_triangle_uqn_state_6q, gjk

n=8
d=2*n
ps = np.linspace(0.0, 1.0, 11)

backend = AerSimulator(device="CPU", method="matrix_product_state")

ds = []
gmns = []
for p in ps:
    qc = random_mipt_1d(n=n, d=d, p=float(p), closed=True)

    qc.save_density_matrix(
        qubits=[0, 1, 2, 3, 4, 5],
        label="final_state",
        pershot=True,
    )

    tqc = transpile(qc, backend)

    # One conditional outcome trajectory for this independently drawn circuit.
    result = backend.run(tqc, shots=1).result()
    sub_m = result.data(0)["final_state"][0]

    psi, rho_init = random_pure_triangle_uqn_state_6q(np.random.default_rng(5678))

    this_gmn = gmn(sub_m, parties=6)
    print(f"p = {p:.2f} GMN: {this_gmn:.8g}")
    gmns.append(this_gmn)

    # print("norm:", np.vdot(psi, psi).real)
    # print("trace:", np.trace(rho_init).real)
    # print("rank:", np.linalg.matrix_rank(rho_init, tol=1e-10))
    # print("shape:", rho_init.shape)


    dist, rho_approx, info = gjk(
        sub_m,
        rho_init,
        seed=1234,
        tol=1e-8,
        max_trials=1_000,
        max_accepts=500
    )

    ds.append(dist)
    print(f"p = {p:.2f} final distance: {dist}")
    print(info)

plt.scatter(ps, ds, color="black")
plt.scatter(ps, gmns, color="red")

trials=331, accepted=260, D=0.432929933133, improvement=1.489e-10        
Final distance: 0.4329299331333366
{'trials': 331, 'accepted': 260, 'rejected_score': 0, 'rejected_degenerate': 0, 'final_d2': 0.1874283270028353, 'final_distance': 0.4329299331333366, 'acceptance_rate': 0.7854984894259819}
trials=357, accepted=220, D=0.546253624626, improvement=1.932e-05        
Final distance: 0.5452478103364496
{'trials': 381, 'accepted': 227, 'rejected_score': 0, 'rejected_degenerate': 0, 'final_d2': 0.297295174676693, 'final_distance': 0.5452478103364496, 'acceptance_rate': 0.5958005249343832}
trials=173, accepted=160, D=0.446643447294, improvement=8.425e-05        
Final distance: 0.4465061097268186
{'trials': 180, 'accepted': 165, 'rejected_score': 0, 'rejected_degenerate': 0, 'final_d2': 0.19936770602337775, 'final_distance': 0.4465061097268186, 'acceptance_rate': 0.9166666666666666}
trials=980, accepted=190, D=0.502212418395, improvement=2.090e-07        
Final distance: 0.49714596841012

KeyboardInterrupt: 

In [1]:
from mipt import *
from gmn import CompiledGMNSDP, gmn_fast

n=8
d=2*n
ps = np.linspace(0.0, 0.7, 8)
reps = 5
subsyst = 4

compiled = CompiledGMNSDP(
    dims=[2] * subsyst,
    formulation="monotone",
    real=True,
)

backend = AerSimulator(device="CPU", method="matrix_product_state")

rhos = {}

print("Starting circuit simulations...")

for p in ps:
    rhos[str(p)] = []
    for _ in range(reps):
        qc = random_mipt_1d(n=n, d=d, p=float(p), closed=True)

        qc.save_density_matrix(
            qubits=list(range(subsyst)),
            label="final_state",
            pershot=True,
        )

        tqc = transpile(qc, backend)

        # One conditional outcome trajectory for this independently drawn circuit.
        result = backend.run(tqc, shots=1).result()
        sub_m = result.data(0)["final_state"][0]
        rhos[str(p)].append(sub_m)

print("Starting GMN computations...")

means = []
errs = []
for p in ps:
    print(f"p={p:.3f}...")
    gmns = []
    for _ in range(reps):

        sub_m = rhos[str(p)].pop()

        score = gmn_fast(
            sub_m,
            formulation="monotone",
        )
        gmns.append(float(score))

    means.append(np.mean(gmns))
    errs.append(np.std(gmns))
    print(f"p={p:.3f}, GMN={means[-1]:.8g} ± {errs[-1]:.3g}")

plt.errorbar(ps, means, yerr=errs, fmt='.', color='black', ecolor='black', capsize=3, elinewidth=1)

Starting circuit simulations...
Starting GMN computations...
p=0.000...


KeyboardInterrupt: 

In [ ]:
from gnme import inflation_score
from mipt import *
from collections import defaultdict
from read_mipt_rho3 import read_rho3_bin


import numpy as np
from collections import defaultdict


def scan_func_by_p(records, func, limit=None):
    """
    Compute mean ± sample stddev of a given function at each p.

    Parameters
    ----------
    records : list[dict]
        Output from read_rho3_bin(...).

    func : callable
        Function taking one 8x8 density matrix and returning a scalar.

    limit : int | None
        Maximum number of matrices to evaluate for each p.
        If None, all matrices are used.

    Returns
    -------
    results : dict[float, dict]
    """

    if limit is not None:
        if not isinstance(limit, int):
            raise TypeError("limit must be an int or None.")
        if limit <= 0:
            raise ValueError("limit must be positive when provided.")

    grouped = defaultdict(list)

    for rec in records:
        grouped[float(rec["p"])].append(rec)

    results = {}

    for p in sorted(grouped):
        group = grouped[p]

        if limit is not None:
            group = group[:limit]

        values = np.asarray(
            [
                float(np.real_if_close(func(rec["rho"])))
                for rec in group
            ],
            dtype=float,
        )

        results[p] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
            "n": int(len(values)),
            "values": values,
        }

    return results

def mixed_ghz_3q(p):
    # Define the basis states
    ket0 = np.array([1, 0])
    ket1 = np.array([0, 1])

    # |000> state = |0> ⊗ |0> ⊗ |0>
    ket000 = np.kron(np.kron(ket0, ket0), ket0)

    # |111> state = |1> ⊗ |1> ⊗ |1>
    ket111 = np.kron(np.kron(ket1, ket1), ket1)

    # 1/sqrt(2) * (|000> + |111>)
    ghz_state = (1 / np.sqrt(2)) * (ket000 + ket111)

    # Density matrix representation: rho = |GHZ><GHZ|
    return (1-p)*np.outer(ghz_state, ghz_state) + p * np.eye(8) / 8


records = read_rho3_bin("rho3.bin")

results = scan_func_by_p(records, inflation_score)

for p, stats in results.items():
    print(
        f"p={p:.3f}: "
        f"F_GHZ = {stats['mean']:.6f} ± {stats['std']:.6f} "
        f"(n={stats['n']})"
    )

p_vals = np.array(sorted(results.keys()))
means = np.array([results[p]["mean"] for p in p_vals])
stds = np.array([results[p]["std"] for p in p_vals])
ns = np.array([results[p]["n"] for p in p_vals])

plt.errorbar(ps, means, yerr=errs, fmt='.', color='black', ecolor='black', capsize=3, elinewidth=1)
plt.xlabel("p")
plt.ylabel("Inflation Score")
plt.title("Mixed GHZ State Inflation Scores")
plt.show()

p=0.00, Inflation Score: 0.99999982
p=0.10, Inflation Score: 0.99998662
p=0.20, Inflation Score: 1.0000088
p=0.30, Inflation Score: 1.0000003
p=0.40, Inflation Score: 1.0000088
p=0.50, Inflation Score: 0.99999802
p=0.60, Inflation Score: 0.99999999
p=0.70, Inflation Score: 0.99999999
p=0.80, Inflation Score: 0.99999999


KeyboardInterrupt: 